# 01b – Khảo sát chất lượng dữ liệu Synthea (EDA)

Chỉ **đọc** `patients.csv`, `encounters.csv`, `providers.csv`, `organizations.csv` trong `data/raw/`, không ghi file nào. Kết quả là cơ sở cho mục 3.2 và 3.3.2 của báo cáo và cho quy tắc làm sạch trong `02b`.

Danh sách kiểm tra: số dòng từng file, khóa ngoại, cột chi phí, loại lượt khám, ngoại lệ chi phí và thời lượng, REASON trống, REVENUE, chuyên khoa của nhà cung cấp.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / 'src' / 'anon_common.py').exists()) / 'src'))
import numpy as np
import pandas as pd
from anon_common import RAW
pd.set_option('display.width', 160)

## 1. Số dòng, số cột và khóa ngoại

In [ ]:
pat = pd.read_csv(RAW / 'patients.csv')
enc = pd.read_csv(RAW / 'encounters.csv', low_memory=False)
prov = pd.read_csv(RAW / 'providers.csv')
org = pd.read_csv(RAW / 'organizations.csv')
print(pd.DataFrame({'dòng': [len(pat), len(enc), len(prov), len(org)], 'cột': [pat.shape[1], enc.shape[1], prov.shape[1], org.shape[1]]},
                   index=['patients', 'encounters', 'providers', 'organizations']))
fk = {'encounters.PATIENT → patients.Id': enc.PATIENT.isin(pat.Id).mean(),
      'encounters.PROVIDER → providers.Id': enc.PROVIDER.isin(prov.Id).mean(),
      'encounters.ORGANIZATION → organizations.Id': enc.ORGANIZATION.isin(org.Id).mean(),
      'providers.ORGANIZATION → organizations.Id': prov.ORGANIZATION.isin(org.Id).mean()}
print(pd.Series(fk).map(lambda v: f'{100 * v:.1f}%').to_string())
print('Bệnh nhân đã mất:', int(pat.DEATHDATE.notna().sum()), '| còn sống:', int(pat.DEATHDATE.isna().sum()))
print('STATE (patients/providers/organizations):', pat.STATE.unique().tolist(), prov.STATE.unique().tolist(), org.STATE.unique().tolist())

## 2. Giai đoạn và loại lượt khám

In [ ]:
start = pd.to_datetime(enc.START, utc=True)
stop = pd.to_datetime(enc.STOP, utc=True)
in_period = start.dt.year >= 2011
print('Từ', start.min(), 'đến', start.max())
print('Trước 2011:', int((~in_period).sum()), f'({100 * (~in_period).mean():.1f}%) | trước 1950:', int((start.dt.year < 1950).sum()),
      '| phần chi phí trước 2011:', round(100 * enc[~in_period].TOTAL_CLAIM_COST.sum() / enc.TOTAL_CLAIM_COST.sum(), 1), '%')
print('Từ 2011:', int(in_period.sum()), 'lượt của', enc[in_period].PATIENT.nunique(), 'bệnh nhân')
print(enc.ENCOUNTERCLASS.value_counts().to_string())
print('Chuyên khoa trong providers.csv:', prov.SPECIALITY.nunique(), '| chuyên khoa của nhà cung cấp thực sự có lượt khám:', prov[prov.Id.isin(enc.PROVIDER)].SPECIALITY.unique().tolist())

## 3. Chi phí theo ENCOUNTERCLASS (từ 2011)

In [ ]:
cost = (enc[in_period].groupby('ENCOUNTERCLASS')
        .agg(n=('TOTAL_CLAIM_COST', 'size'), mean=('TOTAL_CLAIM_COST', 'mean'), median=('TOTAL_CLAIM_COST', 'median'),
             p99=('TOTAL_CLAIM_COST', lambda s: s.quantile(.99)), max=('TOTAL_CLAIM_COST', 'max'),
             base_min=('BASE_ENCOUNTER_COST', 'min'), base_max=('BASE_ENCOUNTER_COST', 'max'))
        .round(1))
print(cost.to_string())
print('\nChi phí gốc có rất ít giá trị khác nhau:', sorted(enc.BASE_ENCOUNTER_COST.round(1).unique())[:6], '...')
print('PAYER_COVERAGE bằng 0:', int((enc.PAYER_COVERAGE == 0).sum()), f'({100 * (enc.PAYER_COVERAGE == 0).mean():.1f}%)')
print('Chi phí bằng 0:', int((enc.TOTAL_CLAIM_COST == 0).sum()), '| trong giai đoạn:', int(((enc.TOTAL_CLAIM_COST == 0) & in_period).sum()))

## 4. Ngoại lệ chi phí: so sánh ba cách

In [ ]:
c = enc[in_period].TOTAL_CLAIM_COST
q1, q3 = c.quantile([.25, .75])
iqr_flag = (c > q3 + 1.5 * (q3 - q1)).sum()
up_n = lo_n = 0
thresholds = {}
for cls, g in enc[in_period].groupby('ENCOUNTERCLASS'):          # IQR trên log(1 + chi phí), tính riêng từng loại lượt khám
    lg_ = np.log1p(g.TOTAL_CLAIM_COST); a, b = lg_.quantile([.25, .75]); k = b - a
    up_n += int((lg_ > b + 1.5 * k).sum()); lo_n += int((lg_ < a - 1.5 * k).sum())
    thresholds[cls] = round(float(np.expm1(b + 1.5 * k)))
p99_flag = pd.Series(False, index=enc.index)
for cls, g in enc[in_period].groupby('ENCOUNTERCLASS'):
    ref = g[g.TOTAL_CLAIM_COST > 0].TOTAL_CLAIM_COST
    p99_flag.loc[g.index] = g.TOTAL_CLAIM_COST > ref.quantile(.99)
print('IQR tuyến tính:', int(iqr_flag), f'({100 * iqr_flag / len(c):.0f}%)')
print('IQR trên log(1+chi phí) theo từng loại: ngưỡng trên (USD) =', thresholds)
print('  gắn cờ phía trên:', up_n, '| phía dưới:', lo_n, '(toàn bộ ở wellness, giá khoảng 270 USD là hợp lệ) | tổng wellness bị gắn cờ:', up_n + lo_n)
print('P99 theo từng loại:', int(p99_flag.sum()), 'dòng, chiếm', round(100 * enc[p99_flag].TOTAL_CLAIM_COST.sum() / c.sum(), 1), '% tổng chi phí')
print('Trong nhóm emergency:', int(p99_flag[enc.ENCOUNTERCLASS.eq('emergency')].sum()), 'ca vượt P99, chiếm',
      round(100 * enc[p99_flag & enc.ENCOUNTERCLASS.eq('emergency')].TOTAL_CLAIM_COST.sum() / enc[in_period & enc.ENCOUNTERCLASS.eq('emergency')].TOTAL_CLAIM_COST.sum()), '% chi phí cấp cứu')

## 5. Thời lượng, REASON trống, REVENUE

In [ ]:
dur = (stop - start).dt.total_seconds() / 60
limit = np.where(enc.ENCOUNTERCLASS.eq('inpatient'), 30 * 1440, 1440)
print('Thời lượng: trung vị', dur.median(), 'phút | lớn nhất', dur.max(), '| nội trú trung vị', dur[enc.ENCOUNTERCLASS.eq('inpatient')].median())
print('Vượt ngưỡng (24 giờ; nội trú 30 ngày):', int((dur > limit).sum()), '| trong giai đoạn:', int(((dur > limit) & in_period).sum()))
print('Lớn nhất trong giai đoạn sau khi bỏ dòng vượt ngưỡng:', dur[in_period & (dur <= limit)].max())
print('REASONDESCRIPTION có giá trị:', int(enc.REASONDESCRIPTION.notna().sum()), 'của', len(enc))
print('Tỷ lệ có lý do theo loại (từ 2011, %):', (enc[in_period].groupby('ENCOUNTERCLASS').REASONDESCRIPTION.apply(lambda s: round(100 * s.notna().mean()))).to_dict())
print('REVENUE bằng 0 ở mọi tổ chức:', bool((org.REVENUE == 0).all()))

## 6. Danh sách vấn đề chất lượng (có số liệu)

| Vấn đề | Số liệu | Xử lý ở `02b` |
|---|---|---|
| Giai đoạn | 26.819 dòng (43,6%) trước 2011 | Cờ `is_in_analysis_period` |
| Chi phí lệch phải | trung vị 786, trung bình 4.897 USD (từ 2011) | Cờ P99 theo loại lượt khám |
| Chi phí bằng 0 | 229 dòng | Cờ `is_zero_cost` |
| Thời lượng cực đoan | 64 dòng vượt ngưỡng | Cờ `is_duration_outlier` |
| REASON trống | 74% | `Not recorded` + cờ `has_reason` |
| REVENUE | 0 ở cả 1.127 tổ chức | Bỏ cột |
| Chuyên khoa | mọi lượt khám gắn GENERAL PRACTICE | Dùng ENCOUNTERCLASS |